## 🎮 Phase 1: The E-Commerce Simulation Environment

In [29]:
import numpy as np

class ECommerceEnvironment:
    """
    Simulates an e-commerce platform where an RL agent tests different discount tiers.
    The goal is to maximize Total Profit, not just the Conversion Rate.
    """
    def __init__(self):
        # The 'Arms' of our Multi-Armed Bandit:
        # Arm 0: 0% discount
        # Arm 1: 10% discount
        # Arm 2: 20% discount
        # Arm 3: 30% discount

        # Hidden true probabilities of a user buying at each discount tier.
        # The RL agent DOES NOT know these values. It must discover them over time.

        self.true_conversionrate = [0.05, 0.15, 0.30, 0.50]

        # Base profit margin for a single product without any discount
        self.base_profit = 100.0

        # Actual profit gained per successful conversion at each discount tier
        self.profits = [
            self.base_profit * 1.0,   # Arm 0: $100 profit
            self.base_profit * 0.9,   # Arm 1: $90 profit
            self.base_profit * 0.8,   # Arm 2: $80 profit
            self.base_profit * 0.7   # Arm 3: $70 profit
        ]

        self.n_arms = len(self.true_conversionrate)

    def step(self, arm_index):
        """
        Simulates a single user interaction with the selected discount (arm).
        
        Args:
            arm_index (int): The index of the discount offered by the RL agent.
            
        Returns:
            conversion (int): 1 if the user bought the item, 0 otherwise.
            revenue (float): The actual profit gained from this interaction.
        """
        # Simulate user behavior using a binomial distribution (coin flip based on true rate)
        conversion = np.random.binomial(1, self.true_conversionrate[arm_index])

        # Calculate the revenue generated (0 if they didn't buy)
        revenue = conversion * self.profits[arm_index]

        return conversion, revenue 

Quick Test

In [30]:
# --- Quick Test ---
env = ECommerceEnvironment()

# Testing Arm 3 (30% discount - highest conversion probability, lowest profit margin)
conversion, revenue = env.step(3)

print(f"Did User buy? {"Yes" if conversion == 1 else "No"}")
print(f"Profit Made: ${revenue}")

Did User buy? Yes
Profit Made: $70.0


## 💻 Phase 2: Thompson Sampling Agent

In [31]:
class ThompsonSamplingAgent:
    """
    An RL Agent that uses Thompson Sampling to maximize Total Profit.
    It learns the underlying conversion rates using Beta distributions, 
    but makes decisions by evaluating Expected Profit.
    """
    def __init__(self, n_arms, profits):
        """
        Args:
            n_arms (int): Number of available discount tiers.
            profits (list or array): The profit margin associated with each arm.
        """
        self.n_arms = n_arms
        self.profits = np.array(profits)

        # Prior distributions: Beta(alpha=1, beta=1) represents a Uniform distribution
        # alpha represents successes (conversions), beta represents failures (no conversions)
        self.alpha = np.ones(n_arms)
        self.beta = np.ones(n_arms)

    def select_arm(self):
        """
        Selects the best discount tier based on Thompson Sampling.
        """
        # 1. Sample an estimated conversion rate for each arm from its Beta distribution
        sampled_conversion_rate = np.random.beta(self.alpha, self.beta)

        # 2. Calculate the expected profit for each arm based on the samples
        expected_profits = sampled_conversion_rate * self.profits

        # 3. Exploit/Explore: Choose the arm with the highest expected profit
        chosen_arm = np.argmax(expected_profits)

        return chosen_arm

    def update(self, arm_index, conversion):
        """
        Updates the Beta distribution parameters based on the environment's feedback.
        
        Args:
            arm_index (int): The discount tier that was offered.
            conversion (int): 1 if the user bought, 0 otherwise.
        """
        if conversion == 1:
            self.alpha[arm_index] += 1  # Success!
        else:
            self.beta[arm_index] += 1   # Failure!

Quick Test

In [32]:
# --- Quick Test ---
# We initialize the agent with 4 arms and the profits from our environment
agent = ThompsonSamplingAgent(env.n_arms, env.profits)

# Agent selects an arm based on its initial belief (alpha=1, beta=1)
chosen_arm = agent.select_arm()
print(f"Agent chose Arm: {chosen_arm}")

# We pass this choice to the environment
conversion, profit = env.step(chosen_arm)

# We update the agent's knowledge based on whether the user bought or not
agent.update(chosen_arm, conversion)

print(f"Updated Alpha Array: {agent.alpha}")
print(f"Updated Beta Array: {agent.beta}")

Agent chose Arm: 1
Updated Alpha Array: [1. 2. 1. 1.]
Updated Beta Array: [1. 1. 1. 1.]
